In [ ]:
from xmlrpc.server import SimpleXMLRPCServer
from xmlrpc.server import SimpleXMLRPCRequestHandler

import datetime
import os
import threading


class RequestHandler(SimpleXMLRPCRequestHandler):
    rpc_paths = ('/RPC2',)

server = SimpleXMLRPCServer(("localhost", 8018),
                            requestHandler=RequestHandler,
                            allow_none=True)
server.register_introspection_functions()

# ---------- Настройки ----------
LOG_FILE = 'log.csv'
MAX_RECORDS = 20          # п.5 — ограничение размера журнала
_lock = threading.Lock()  # защита от одновременной записи


def _count_records():
    """Сколько строк сейчас в log.csv."""
    if not os.path.exists(LOG_FILE):
        return 0
    with open(LOG_FILE, 'r', encoding='utf-8') as f:
        return sum(1 for _ in f)


def _rotate():
    """п.5 — переименовать log.csv в log_YYYYMMDD_hhmmss.csv."""
    if os.path.exists(LOG_FILE):
        stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        os.rename(LOG_FILE, f'log_{stamp}.csv')
        print(f"[ROTATE] log.csv -> log_{stamp}.csv")


# ---------- Запись в журнал ----------
def add_log(sname):
    """
    Формат строки: <тип_события>,<YYYY-MM-DD HH:MM:SS>
    При превышении MAX_RECORDS — ротация (п.5).
    """
    with _lock:
        if _count_records() >= MAX_RECORDS:
            _rotate()
        with open(LOG_FILE, 'a', encoding='utf-8') as f:
            f.write(str(sname) + ',' +
                    datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S") + '\n')
    return True

server.register_function(add_log, 'add_log')


# ---------- п.6 — чтение журнала с фильтрами ----------
def get_log(event_type=None, dt_from=None, dt_to=None):
    """
    event_type      — строка ('ping', 'sum', ...) или None
    dt_from, dt_to  — границы в формате 'YYYY-MM-DD HH:MM:SS' или None
    Возвращает список [тип, дата].
    """
    rows = []
    if not os.path.exists(LOG_FILE):
        return rows
    with open(LOG_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(',', 1)
            if len(parts) != 2:
                continue
            ev, ts = parts[0], parts[1]

            if event_type and ev != event_type:
                continue

            try:
                t = datetime.datetime.strptime(ts, "%Y-%m-%d %H:%M:%S")
            except ValueError:
                continue

            if dt_from:
                if t < datetime.datetime.strptime(dt_from, "%Y-%m-%d %H:%M:%S"):
                    continue
            if dt_to:
                if t > datetime.datetime.strptime(dt_to, "%Y-%m-%d %H:%M:%S"):
                    continue

            rows.append([ev, ts])
    return rows

server.register_function(get_log, 'get_log')


print("Listening on port 8018...")
server.serve_forever()

Listening on port 8018...


In [ ]:
from xmlrpc.server import SimpleXMLRPCServer
from xmlrpc.server import SimpleXMLRPCRequestHandler

import datetime
import os
import threading


class RequestHandler(SimpleXMLRPCRequestHandler):
    rpc_paths = ('/RPC2',)


class ReusableServer(SimpleXMLRPCServer):
    """Подкласс с allow_reuse_address=True ДО __init__ (иначе не работает)."""
    allow_reuse_address = True


server = ReusableServer(("localhost", 8018),
                        requestHandler=RequestHandler,
                        allow_none=True)
server.register_introspection_functions()

# ---------- Настройки ----------
LOG_FILE = 'log.csv'
MAX_RECORDS = 20
_lock = threading.Lock()


# ================= ДИАГНОСТИКА =================

def ping_stats():
    """Быстрая проверка, что клиент попал именно в этот сервер."""
    return "stats_server_ok"

server.register_function(ping_stats, 'ping_stats')


def list_stats_methods():
    """Возвращает список зарегистрированных методов."""
    return server.system_listMethods()

server.register_function(list_stats_methods, 'list_stats_methods')


# ================= ЖУРНАЛ =================

def _count_records():
    if not os.path.exists(LOG_FILE):
        return 0
    with open(LOG_FILE, 'r', encoding='utf-8') as f:
        return sum(1 for _ in f)


def _rotate():
    """п.5 — переименовать log.csv в log_YYYYMMDD_hhmmss.csv."""
    if os.path.exists(LOG_FILE):
        stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        new_name = f'log_{stamp}.csv'
        os.rename(LOG_FILE, new_name)
        print(f"[ROTATE] log.csv -> {new_name}")


def add_log(sname):
    """
    Формат строки: <тип_события>,<YYYY-MM-DD HH:MM:SS>
    При превышении MAX_RECORDS — ротация (п.5).
    """
    with _lock:
        if _count_records() >= MAX_RECORDS:
            _rotate()
        with open(LOG_FILE, 'a', encoding='utf-8') as f:
            f.write(str(sname) + ',' +
                    datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S") + '\n')
    return True

server.register_function(add_log, 'add_log')


# ================= ЧТЕНИЕ С ФИЛЬТРАМИ =================

def get_log(event_type=None, dt_from=None, dt_to=None):
    """
    event_type      — строка ('ping', 'sum', ...) или None
    dt_from, dt_to  — 'YYYY-MM-DD HH:MM:SS' или None
    Возвращает список кортежей (тип, дата).
    """
    rows = []
    if not os.path.exists(LOG_FILE):
        return rows

    with open(LOG_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split(',', 1)
            if len(parts) != 2:
                continue
            ev, ts = parts[0].strip(), parts[1].strip()

            if event_type and ev != event_type:
                continue

            try:
                t = datetime.datetime.strptime(ts, "%Y-%m-%d %H:%M:%S")
            except ValueError:
                continue

            if dt_from:
                if t < datetime.datetime.strptime(dt_from, "%Y-%m-%d %H:%M:%S"):
                    continue
            if dt_to:
                if t > datetime.datetime.strptime(dt_to, "%Y-%m-%d %H:%M:%S"):
                    continue

            rows.append((ev, ts))   # кортеж вместо списка — надёжнее в XML-RPC

    return rows

server.register_function(get_log, 'get_log')


# ================= СТАРТ =================
print("=" * 50)
print("STATS SERVER STARTED")
print(f"  port       : 8018")
print(f"  log file   : {os.path.abspath(LOG_FILE)}")
print(f"  max records: {MAX_RECORDS}")
print(f"  methods    : {server.system_listMethods()}")
print("=" * 50)
print("Listening on port 8018...")

server.serve_forever()

STATS SERVER STARTED
  port       : 8018
  log file   : /home/ksenia/spacenet_building_detection/PSOI/lab_2/metoda/log.csv
  max records: 20
  methods    : ['add_log', 'get_log', 'list_stats_methods', 'ping_stats', 'system.listMethods', 'system.methodHelp', 'system.methodSignature']
Listening on port 8018...


127.0.0.1 - - [28/Sep/2026 22:51:49] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [28/Sep/2026 22:51:49] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [28/Sep/2026 22:51:49] "POST /RPC2 HTTP/1.1" 200 -
